In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess


def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py

In [ ]:
from pong import fresh_params, run_config
from util import eval_pong_ql_score, plot_dqn_metrics

In [ ]:
cfg = {
    "lr": 1e-3,
    "decay_dur": 100_000,
    "hidden_dim": 64,
    "num_layers": 1,
}
total_steps = 500_000

ep_rets, losses, params = run_config(
    cfg, total_steps, fresh_params(cfg["hidden_dim"], cfg["num_layers"])
)
mean_last50 = sum(ep_rets[-50:]) / min(50, len(ep_rets))
print(f"Done. {len(ep_rets)} episodes, mean last-50 return: {mean_last50:.1f}")

In [ ]:
plot_dqn_metrics(losses, ep_rets)

In [ ]:
mean_score, n_eps = eval_pong_ql_score(
    params,
    cfg["hidden_dim"],
    cfg["num_layers"],
    num_eval_episodes=100,
    batch_size=20,
    seed=123,
    show_progress=True,
)
print(f"Mean episode score over {n_eps} episodes: {mean_score:.1f}  (max possible: ~999)")

In [ ]:
%%bash
# Build Pong visualization
set -euo pipefail

cd ../ts
pnpm i
pnpm run build:anywidget-pong


In [ ]:
from pathlib import Path
from util import as_f32, write_safetensors
from pong import make_model

model = make_model(cfg["hidden_dim"], cfg["num_layers"], params)

# Transpose kernels: (in_dim, out_dim) -> (out_dim, in_dim).
tensors = {}
for i in range(model.num_layers):
    layer = getattr(model, f"layer_{i}")
    tensors[f"w{i}"] = as_f32(layer.kernel).T
    tensors[f"b{i}"] = as_f32(layer.bias).reshape(-1)

out_path = Path("pong-weights.safetensors")
write_safetensors(out_path, tensors)

print(f"Wrote {out_path}")
for name, arr in tensors.items():
    print(f"  {name}: {tuple(arr.shape)}")

In [ ]:
import base64
from IPython.display import display
from pong import PongVisualization

weights_bytes = Path("pong-weights.safetensors").read_bytes()
weights_b64 = base64.b64encode(weights_bytes).decode("ascii")

display(PongVisualization(weights_base64=weights_b64))